
# Difficulty × Problem Type Heatmap (Single-Model)

This slim notebook focuses on one thing: **plot a heatmap** of results by *difficulty* and *problem type*, using only the model actually present in the data.
It avoids multi-model smoke tests, provider calls, and scripts. It also guards against the common `is_correct`/column missing issues observed.


# Math Problem Performance Test

This notebook tests the model on problems from problems.csv and analyzes performance by problem type and difficulty.

## Setup and Model Loading

## Load Problems Dataset

## Helper Functions

## Smoke test

## Run Performance Test

Evaluate the model on a subset of problems. Start with a small sample to test, then increase.

## Analysis by Topic and Difficulty

## Heatmap: Performance by Topic and Difficulty

## Additional Visualizations

## Capture Hidden States (Sample)

Capture hidden states for a sample of problems to analyze later. This can help understand what the model is learning.

## Save Results

## Example: View Some Results

In [ ]:

# Minimal imports
import math, re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [ ]:

# === Helpers: robust column detection, correctness, and heatmap plotting ===
import math, re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def detect_column(df: pd.DataFrame, candidates, default=None):
    cols = [c for c in candidates if c in df.columns]
    return cols[0] if cols else default

_num_re = re.compile(r"^[\+\-]?(?:\d+\.?\d*|\.\d+)(?:[eE][\+\-]?\d+)?$")

def _strip_latex(s: str) -> str:
    if s is None:
        return ""
    s = str(s).strip()
    s = re.sub(r"\\boxed\{(.+?)\}", r"\1", s)
    s = re.sub(r"\\\((.+?)\\\)", r"\1", s)
    s = re.sub(r"\\\[(.+?)\\\]", r"\1", s)
    s = s.replace("$", "")
    return s.strip()

def _as_number(s: str):
    if s is None:
        return None
    s = _strip_latex(str(s)).strip().replace(",", "")
    return float(s) if _num_re.match(s) else None

def normalize_text(s: str) -> str:
    if s is None:
        return ""
    s = _strip_latex(s)
    s = re.sub(r"\s+", " ", s).strip().lower()
    return s

def answers_match(gold, pred, rel=1e-6, abs_=1e-9) -> bool:
    gnum, pnum = _as_number(gold), _as_number(pred)
    if gnum is not None and pnum is not None:
        return math.isclose(gnum, pnum, rel_tol=rel, abs_tol=abs_)
    return normalize_text(pred) == normalize_text(gold)

def ensure_eval_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    topic_col = detect_column(out, ['problem_type','topic','type','category','subject'], default=None)
    if topic_col is None:
        out['problem_type'] = 'Unknown'
        topic_col = 'problem_type'

    diff_col = detect_column(out, ['difficulty','level','tier'], default=None)
    if diff_col is None:
        out['difficulty'] = 'Unknown'
        diff_col = 'difficulty'

    # Build is_correct if missing
    if 'is_correct' not in out.columns:
        # pick gold/pred columns
        gold_col = detect_column(out, ['answer','gold','expected','ground_truth','label'], default=None)
        pred_col = detect_column(out, ['model_answer','prediction','pred','model_output','response','answer_pred'], default=None)
        if gold_col is not None and pred_col is not None:
            out['is_correct'] = out.apply(lambda r: answers_match(r.get(gold_col,''), r.get(pred_col,'')), axis=1)
        else:
            # mark as False but continue; heatmap will then work for counts
            out['is_correct'] = False

    return out, topic_col, diff_col

def select_single_model(df: pd.DataFrame) -> (pd.DataFrame, str):
    model_col = detect_column(df, ['model','model_name','provider_model','engine'], default=None)
    if model_col is None:
        return df, None
    vc = df[model_col].value_counts(dropna=True)
    if vc.empty:
        return df, None
    chosen = vc.index[0]
    return df[df[model_col] == chosen].copy(), chosen

def plot_accuracy_heatmap(df: pd.DataFrame, topic_col: str, diff_col: str, title_suffix=""):
    # Build accuracy pivot (mean of is_correct)
    piv = df.pivot_table(index=topic_col, columns=diff_col, values='is_correct', aggfunc='mean', fill_value=0.0)
    # Ensure stable order
    piv = piv.sort_index().reindex(sorted(piv.columns), axis=1)

    # Plot with matplotlib (no seaborn, single figure, default colors)
    fig, ax = plt.subplots(figsize=(max(6, len(piv.columns)*0.9), max(4, len(piv.index)*0.5)))
    im = ax.imshow(piv.values, aspect='auto')  # default colormap

    # Ticks/labels
    ax.set_xticks(np.arange(len(piv.columns)))
    ax.set_yticks(np.arange(len(piv.index)))
    ax.set_xticklabels(list(piv.columns), rotation=45, ha='right')
    ax.set_yticklabels(list(piv.index))
    ax.set_xlabel(diff_col.title())
    ax.set_ylabel(topic_col.replace('_',' ').title())
    ax.set_title(f"Accuracy Heatmap by {topic_col.replace('_',' ').title()} × {diff_col.title()}" + (f" — {title_suffix}" if title_suffix else ""))

    # Annotate with percentage text
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            val = piv.values[i, j]
            ax.text(j, i, f"{val*100:.0f}%", ha="center", va="center", fontsize=8)

    fig.tight_layout()
    plt.show()


### Data assembly (from original notebook)
The following cell is preserved because it assigns `eval_df`. If it depends on earlier cells you removed, adapt it to read from your saved results (e.g., CSV).

In [ ]:
import math

SEED = 42
SAMPLE_SIZE = 15  # total number of problems you actually want

required = {'question', 'answer', 'topic', 'difficulty'}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing columns in df: {sorted(missing)}")

# stratified sample up to SAMPLE_SIZE total
group_cols = ['topic', 'difficulty']
gb = df.groupby(group_cols, dropna=False)
num_groups = len(gb)

if SAMPLE_SIZE >= len(df):
    eval_df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
else:
    # allocate 1 per group until we hit SAMPLE_SIZE, else proportional
    if SAMPLE_SIZE <= num_groups:
        # pick SAMPLE_SIZE groups at random, 1 from each
        chosen_groups = (
            gb.size()
              .sample(n=SAMPLE_SIZE, random_state=SEED)
              .index
        )
        parts = []
        for gkey in chosen_groups:
            g = gb.get_group(gkey)
            parts.append(g.sample(n=1, random_state=SEED))
        eval_df = pd.concat(parts, ignore_index=True)
    else:
        # proportional quotas (at least 1 if group non-empty)
        sizes = gb.size()
        weights = sizes / sizes.sum()
        base = (weights * SAMPLE_SIZE).round().astype(int)
        # ensure at least 1 where possible
        base = base.mask((sizes > 0) & (base == 0), 1)
        # adjust to exact total
        diff = SAMPLE_SIZE - base.sum()
        if diff != 0:
            order = (weights.sort_values(ascending=(diff < 0))
                              .index.tolist())
            for gkey in order:
                if diff == 0:
                    break
                step = 1 if diff > 0 else -1
                new_val = base.loc[gkey] + step
                if 0 <= new_val <= sizes.loc[gkey]:
                    base.loc[gkey] = new_val
                    diff -= step
        # sample per group
        parts = []
        for gkey, k in base.items():
            if k > 0:
                g = gb.get_group(gkey)
                parts.append(g.sample(n=min(k, len(g)), random_state=SEED))
        eval_df = pd.concat(parts, ignore_index=True)

print(f"Evaluating on {len(eval_df)} problems...")
if set(group_cols).issubset(eval_df.columns):
    print("Distribution:\n", eval_df.value_counts(group_cols).sort_index())
else:
    print("Skipping distribution print; stratifying columns not present in eval_df.")


Evaluating on 15 problems...
Distribution:
 topic                   difficulty
Algebra                 Level 1       1
                        Level 2       1
                        Level 3       1
                        Level 4       1
                        Level 5       1
Counting & Probability  Level 1       1
                        Level 2       1
                        Level 3       1
                        Level 4       1
                        Level 5       1
Number Theory           Level 1       1
                        Level 2       1
                        Level 3       1
                        Level 4       1
                        Level 5       1
Name: count, dtype: int64


In [ ]:

# === Heatmap analysis ===
# Expectation: a DataFrame named `eval_df` exists at this point.
try:
    _ = eval_df
except NameError:
    raise RuntimeError("`eval_df` is not defined. Please run the data assembly cell above or create `eval_df` before this cell.")

eval_df_single, chosen_model = select_single_model(eval_df)
if chosen_model:
    print(f"Using model: {chosen_model}")

eval_df_fixed, topic_col, diff_col = ensure_eval_columns(eval_df_single)

print(f"Detected topic/problem-type column: {topic_col}")
print(f"Detected difficulty column: {diff_col}")
print(f"Rows: {len(eval_df_fixed)} | is_correct present: {'is_correct' in eval_df_fixed.columns}")

plot_accuracy_heatmap(eval_df_fixed, topic_col, diff_col, title_suffix=(chosen_model or ''))


### Errors observed in the original notebook (for context)

- `KeyError: 'Column not found: is_correct'`